In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve
)

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("cleaned_data.csv")

In [3]:
X = df.drop(columns=["spam_score","label"])

X = pd.get_dummies(
    X,
    columns=["subject","sender_domain"],
    drop_first=True
)

y = df["label"].map({
    "ham":0,
    "spam":1
})

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [6]:
tree = DecisionTreeClassifier(
    random_state=42
)

tree.fit(
    X_train_scaled,
    y_train
)

train_acc = accuracy_score(
    y_train,
    tree.predict(X_train_scaled)
)

test_acc = accuracy_score(
    y_test,
    tree.predict(X_test_scaled)
)

print("Training Accuracy :",train_acc)

print("Testing Accuracy :",test_acc)

Training Accuracy : 1.0
Testing Accuracy : 1.0


In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

controlled_tree.fit(X_train_scaled, y_train)

train_acc2 = accuracy_score(
    y_train,
    controlled_tree.predict(X_train_scaled)
)

test_acc2 = accuracy_score(
    y_test,
    controlled_tree.predict(X_test_scaled)
)

print("Training Accuracy :", train_acc2)
print("Testing Accuracy :", test_acc2)

Training Accuracy : 1.0
Testing Accuracy : 1.0


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Gini Tree
gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

# Entropy Tree
entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

gini_tree.fit(X_train_scaled, y_train)
entropy_tree.fit(X_train_scaled, y_train)

gini_acc = accuracy_score(
    y_test,
    gini_tree.predict(X_test_scaled)
)

entropy_acc = accuracy_score(
    y_test,
    entropy_tree.predict(X_test_scaled)
)

print("Gini Accuracy:", gini_acc)
print("Entropy Accuracy:", entropy_acc)

Gini Accuracy: 1.0
Entropy Accuracy: 1.0


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train_scaled, y_train)

train_acc = accuracy_score(
    y_train,
    rf.predict(X_train_scaled)
)

test_acc = accuracy_score(
    y_test,
    rf.predict(X_test_scaled)
)

rf_prob = rf.predict_proba(X_test_scaled)[:,1]

auc = roc_auc_score(
    y_test,
    rf_prob
)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)
print("ROC-AUC:", auc)

Training Accuracy: 1.0
Testing Accuracy: 1.0
ROC-AUC: 1.0


In [10]:
importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": rf.feature_importances_

})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance.head(5))

                  Feature  Importance
3        num_exclamations    0.285543
2               num_links    0.272256
4  capital_letter_percent    0.174283
5               has_offer    0.104605
1            email_length    0.066764


In [11]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb.fit(X_train_scaled, y_train)

train_acc = accuracy_score(
    y_train,
    gb.predict(X_train_scaled)
)

test_acc = accuracy_score(
    y_test,
    gb.predict(X_test_scaled)
)

gb_prob = gb.predict_proba(X_test_scaled)[:,1]

auc = roc_auc_score(
    y_test,
    gb_prob
)

print("Training Accuracy:", train_acc)
print("Testing Accuracy:", test_acc)
print("ROC-AUC:", auc)

Training Accuracy: 1.0
Testing Accuracy: 1.0
ROC-AUC: 0.9999999999999999


In [12]:
# Bottom 5 least important features
low_features = importance.tail(5)["Feature"].tolist()

print("Lowest 5 Features:")
print(low_features)

# Remove lowest features
X_train_reduced = X_train.drop(columns=low_features)
X_test_reduced = X_test.drop(columns=low_features)

# Scale reduced data
scaler2 = StandardScaler()

X_train_red_scaled = scaler2.fit_transform(X_train_reduced)
X_test_red_scaled = scaler2.transform(X_test_reduced)

# Train Random Forest again
rf2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf2.fit(X_train_red_scaled, y_train)

full_auc = roc_auc_score(
    y_test,
    rf.predict_proba(X_test_scaled)[:,1]
)

reduced_auc = roc_auc_score(
    y_test,
    rf2.predict_proba(X_test_red_scaled)[:,1]
)

print("Full Model AUC:", full_auc)
print("Reduced Model AUC:", reduced_auc)

Lowest 5 Features:
['sender_domain_gmail.com', 'sender_domain_offers.org', 'sender_domain_outlook.com', 'sender_domain_promo.net', 'sender_domain_yahoo.com']
Full Model AUC: 1.0
Reduced Model AUC: 1.0


In [13]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=20,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
}

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc"
    )

    print("\n", name)
    print("Mean AUC :", scores.mean())
    print("Std AUC :", scores.std())


 Logistic Regression
Mean AUC : 1.0
Std AUC : 0.0

 Decision Tree
Mean AUC : 0.9935707678075856
Std AUC : 0.008534235075011724

 Random Forest
Mean AUC : 1.0
Std AUC : 0.0

 Gradient Boosting
Mean AUC : 0.9978260869565216
Std AUC : 0.0043478260869565305


In [14]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest CV Score:")
print(grid.best_score_)

Best Parameters:
{'randomforestclassifier__max_depth': 5, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 50}

Best CV Score:
1.0


In [15]:
from sklearn.metrics import roc_auc_score
import pandas as pd

best_pipeline = grid.best_estimator_

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results = []

for f in fractions:

    n = int(f * len(X_train))

    X_sub = X_train.iloc[:n]
    y_sub = y_train.iloc[:n]

    best_pipeline.fit(X_sub, y_sub)

    train_prob = best_pipeline.predict_proba(X_sub)[:,1]
    test_prob = best_pipeline.predict_proba(X_test)[:,1]

    train_auc = roc_auc_score(y_sub, train_prob)
    test_auc = roc_auc_score(y_test, test_prob)

    results.append([f, train_auc, test_auc])

learning_curve = pd.DataFrame(
    results,
    columns=[
        "Training Fraction",
        "Training AUC",
        "Test AUC"
    ]
)

print(learning_curve)

   Training Fraction  Training AUC  Test AUC
0                0.2           1.0       1.0
1                0.4           1.0       1.0
2                0.6           1.0       1.0
3                0.8           1.0       1.0
4                1.0           1.0       1.0


In [16]:
import joblib

joblib.dump(best_pipeline, "best_model.pkl")

print("Model Saved Successfully!")

Model Saved Successfully!


In [17]:
import joblib

# Load saved model
loaded_model = joblib.load("best_model.pkl")

# Take first 2 rows as sample
sample_data = X.iloc[:2]

# Predict
predictions = loaded_model.predict(sample_data)

print("Predictions:")
print(predictions)

Predictions:
[1 1]
